<figure>
  <img src="https://raw.githubusercontent.com/shadowkshs/DimABSA2026/refs/heads/main/banner.png" width="100%">
</figure>

In [1]:
AUGMENTED = "contextual-augmentation/results/retry20/eng_laptop_train_alltasks_contextual_aug_retry20_original_plus_augmented.jsonl"

In [2]:
# %load_ext autoreload
# %autoreload 2

import json, yaml
from typing import List, Dict
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import logging
import sys
import os

import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import sentencepiece

from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

In [3]:
if "google.colab" in sys.modules :
    REPO_PATH = Path("/content/NLP_semeval26_task3_DimASR")

    if REPO_PATH.exists():
        %rm -rf "/content/NLP_semeval26_task3_DimASR"
        !git clone "https://github.com/Projet-NLP-UdeS/NLP_semeval26_task3_DimASR.git"
    else :
        !git clone "https://github.com/Projet-NLP-UdeS/NLP_semeval26_task3_DimASR.git"

    %cd "/content/NLP_semeval26_task3_DimASR"
    !git checkout colab_outputs
    sys.path.insert(0, str(REPO_PATH))

from src.data import *
from src.eval import *
from src.models.svr import run_svr_baseline
from src.models.bert import TransformerVARegressor
from src.models.ensemble import (
    AverageEnsemble
)

Cloning into 'NLP_semeval26_task3_DimASR'...
remote: Enumerating objects: 326, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 326 (delta 36), reused 104 (delta 23), pack-reused 204 (from 1)
Receiving objects: 100% (326/326), 2.91 MiB | 7.79 MiB/s, done.
Resolving deltas: 100% (132/132), done.
/content/NLP_semeval26_task3_DimASR
Branch 'colab_outputs' set up to track remote branch 'colab_outputs' from 'origin'.
Switched to a new branch 'colab_outputs'


In [4]:
log_format = "%(asctime)s | %(levelname)s | %(message)s \n"
logging.basicConfig(
    level=logging.INFO,
    format=log_format,
    force=True,
)

logger = logging.getLogger()
fh = logging.FileHandler("outputs/results/log.txt")
fh.setFormatter(logging.Formatter(log_format))
logger.addHandler(fh)

logging.info("This shows in notebook and goes to file")

2026-04-14 04:19:08,818 | INFO | This shows in notebook and goes to file 



In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"Will be using {device} device.")
# device = torch.device("cpu") # force

# Set testing filter
# for faster training testing
testing = True # (device.type == "cpu")
if testing: (logging.info(f"Will be using a lighter training configuration, not suitable for final results."))

2026-04-14 04:19:08,821 | INFO | Will be using cuda device. 

2026-04-14 04:19:08,822 | INFO | Will be using a lighter training configuration, not suitable for final results. 



### Step 1: Load datasets and configuration


In [6]:
subtask = "subtask_1"
task = "task1"
lang = "eng"
domain = "restaurant"

!pwd
if AUGMENTED is not None :
    logging.info("Will be using local augmented dataset")
    train_raw = load_jsonl(AUGMENTED)
else :
    logging.info("Will be using remote default dataset")
    train_url = (f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/"
                 f"task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_alltasks.jsonl")
    train_raw = load_jsonl_url(train_url)

predict_url = (f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/"
               f"task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl")
predict_raw = load_jsonl_url(predict_url)

train_df = jsonl_to_df(train_raw)
predict_df = jsonl_to_df(predict_raw)

train_df = train_df.sample(100) if testing else train_df

# split 10% for dev
train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=42)


with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

models = config["models"]
logging.info(json.dumps(models, indent=2))

/content/NLP_semeval26_task3_DimASR


2026-04-14 04:19:08,935 | INFO | Will be using local augmented dataset 

2026-04-14 04:19:09,215 | INFO | [
  {
    "name": "distilbert-base-uncased-finetuned-sst-2-english",
    "nickname": "baby_bert",
    "type": "transformer",
    "lr": "2e-5",
    "epochs": 4,
    "batch_size": 32,
    "dropout": 0.1
  },
  {
    "name": "bert-base-multilingual-cased",
    "nickname": "bert_base",
    "type": "transformer",
    "lr": "2e-5",
    "epochs": 4,
    "batch_size": 32,
    "dropout": 0.1
  },
  {
    "name": "microsoft/deberta-v3-base",
    "nickname": "deberta_base",
    "type": "transformer",
    "lr": "2e-5",
    "epochs": 4,
    "batch_size": 32,
    "dropout": 0.1,
    "max_len": 128
  },
  {
    "name": "FacebookAI/roberta-base",
    "nickname": "roberta_base",
    "type": "transformer",
    "lr": "2e-5",
    "epochs": 4,
    "batch_size": 16,
    "dropout": 0.1
  }
] 



### Display the dataframe

In [7]:
from IPython.display import display, Markdown

display(Markdown(f"### {subtask}_{lang}_{domain} train_df"))
display(train_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} dev_df"))
display(dev_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} predict_df"))
display(predict_df.head())

### subtask_1_eng_restaurant train_df

,Aspect,ID,Text,Valence,Arousal
4052,keyboard / mousepad,laptop_quad_train_1799,the keyboard / mousepad isn ' t super comforta...,3.67,6.67
3969,asus chromebook,laptop_quad_train_1735,i purchased this asus chromebook in may of 201...,7.75,7.75
2939,hd screen,laptop_quad_train_973,"saving the best for last , the full hd screen ...",7.38,7.62
4525,monitor,laptop_quad_train_2147,monitor went out 35 days after receiving .,4.75,5.38
1170,NULL,laptop_quad_test_536,when it is functioning it works great .,7.50,7.67


### subtask_1_eng_restaurant dev_df

,Aspect,ID,Text,Valence,Arousal
597,NULL,laptop_quad_test_121,this was purchased for my son to download his ...,5.38,5.38
6388,NULL,laptop_quad_train_12_ctxaug_1,i just got this thing today & was really disap...,2.88,6.50
4043,NULL,laptop_quad_train_1794,"it ' s fast , but the whole thing is wearing o...",6.88,6.88
2263,computer,laptop_quad_train_480,the computer runs great .,7.25,7.50
5656,NULL,laptop_quad_train_2863,it ' s now been totally reliable for half a ye...,7.33,7.17


### subtask_1_eng_restaurant predict_df

,Aspect,VA,ID,Text,Valence,Arousal
0,diner food,7.25#6.75,rest26_aspect_va_dev_1,Great diner food and breakfast is served all day,7.25,6.75
1,breakfast,7.25#6.75,rest26_aspect_va_dev_1,Great diner food and breakfast is served all day,7.25,6.75
2,food,7.50#7.75,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.50,7.75
3,drinks,7.50#7.50,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.50,7.50
4,service,7.75#7.75,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.75,7.75


### Step 2 : Train all models in config.yaml

In [8]:
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    checkpoint_dir = "/content/drive/MyDrive/UdS-IFT714-checkpoints"
else:
    checkpoint_dir = "outputs/checkpoints"

os.makedirs(checkpoint_dir, exist_ok=True)


def save_model_checkpoint(
    checkpoint_dir=checkpoint_dir,
    nickname="bert_model_default",
    epoch=4,
    model=None,
    optimizer=None,
    train_loss=None,
    val_loss=None,
    lr=None,
    epochs=None,
    batch_size=None,
    dropout=None,
    max_len=None
    ):
    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"{nickname}_{epoch+1}_{epochs}_last.pt"
    )

    torch.save(
        {
            "model_name": nickname,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": lr,
            "batch_size": batch_size,
            "dropout": dropout,
            "max_len": max_len,
        },
        checkpoint_path,
    )

    logging.info(f"Checkpoint saved: {checkpoint_path}")

def load_model_checkpoint(
    checkpoint_dir=checkpoint_dir,
    nickname=None,
    epoch=None,
    epochs=None,
    model=None,
    optimizer=None,
    device="cpu",
    ):

    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"{nickname}_{epoch+1}_{epochs}_last.pt"
    )

    checkpoint = torch.load(checkpoint_path, map_location=device)

    # # Load model weights
    # if model is not None:
    #     model.load_state_dict(checkpoint["model_state_dict"])

    # # Load optimizer state (optional)
    # if optimizer is not None and "optimizer_state_dict" in checkpoint:
    #     optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    logging.info(f"Checkpoint loaded: {checkpoint_path}")

    return checkpoint


Mounted at /content/drive


In [9]:
model_results = {} # Pour stocker les scores finaux
trained_models = {}
ensemble = dev_df

for arch in models:
    current_model = arch["name"]
    model_type = arch["type"]
    current_nickname = arch.get("nickname", current_model)

    print(f"\n{'='*80}")
    print(f"ENTRAÎNEMENT DU MODÈLE : {current_model}")
    print(f"{'='*80}")

    # Pipline Deep learning
    if model_type == "transformer":

        current_lr = float(arch["lr"])
        current_epochs = arch["epochs"]
        current_dropout = arch["dropout"]

        current_batch_size = arch["batch_size"] if not testing else 1

        tokenizer = AutoTokenizer.from_pretrained(current_model)
        default_max_len = tokenizer.model_max_length if tokenizer.model_max_length < 1025 else 128
        current_max_len = int(arch.get("max_len", default_max_len))
        logging.info(f"Current maximum token length is {current_max_len}")

        print(f"Paramètres : LR={current_lr}, Epochs={current_epochs}, Batch={current_batch_size}, Dropout={current_dropout}")

        # Création des DataLoaders
        train_dataset = VADataset(train_df, tokenizer, max_len=current_max_len)
        dev_dataset = VADataset(dev_df, tokenizer, max_len=current_max_len)

        train_loader = DataLoader(train_dataset, batch_size=current_batch_size, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=current_batch_size, shuffle=False)

        # Initialisation du modèle
        model = TransformerVARegressor(current_model_name=current_model, dropout=current_dropout).to(device).float()
        optimizer = torch.optim.AdamW(model.parameters(), lr=current_lr)
        loss_fn = nn.MSELoss()

        # Entraînement du modèle
        for epoch in range(current_epochs):
            train_loss = model.train_epoch(train_loader, optimizer, loss_fn, device)
            val_loss = model.eval_epoch(dev_loader, loss_fn, device)
            logging.info(f"Epoch {epoch+1}/{current_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

            save_model_checkpoint(
                checkpoint_dir=checkpoint_dir,
                nickname=current_nickname,
                model=model,
                optimizer=optimizer,
                epoch=epoch,
                train_loss=train_loss,
                val_loss=val_loss,
                lr=current_lr,
                epochs=current_epochs,
                batch_size=current_batch_size,
                dropout=current_dropout,
                max_len=current_max_len
            )

        # Évaluation du modèle sur le Dev Set
        pred_v, pred_a, gold_v, gold_a = get_prd(model, dev_loader, type="dev")
        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_nickname] = eval_score
        trained_models[current_nickname] = model

        # Saving predictions for ensemble learning
        ensemble = predict_to_dataframe(
            model, dev_loader, ensemble,
            pred_v_col = f"{current_nickname}_valence",
            pred_a_col = f"{current_nickname}_arousal"
        )

    # Pipline Machine Learning
    elif model_type == "sklearn":

        max_features = arch["max_features"]

        pred_v, pred_a, gold_v, gold_a = run_svr_baseline(train_df, dev_df, max_features=max_features)

        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_nickname] = eval_score

        # ensemble = predict_to_dataframe(
        #     model, dev_loader, ensemble,
        #     pred_v_col = f"{current_model}_valence",
        #     pred_a_col = f"{current_model}_arousal"
        # )


ENTRAÎNEMENT DU MODÈLE : distilbert-base-uncased-finetuned-sst-2-english


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
2026-04-14 04:19:29,111 | INFO | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json "HTTP/1.1 200 OK" 

2026-04-14 04:19:29,223 | INFO | HTTP Request: GET https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json "HTTP/1.1 200 OK" 



config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

2026-04-14 04:19:29,361 | INFO | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK" 

2026-04-14 04:19:29,471 | INFO | HTTP Request: GET https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK" 



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Paramètres : LR=2e-05, Epochs=4, Batch=1, Dropout=0.1


2026-04-14 04:26:14,695 | INFO | HTTP Request: HEAD https://huggingface.co/FacebookAI/roberta-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found" 

2026-04-14 04:26:14,811 | INFO | HTTP Request: HEAD https://huggingface.co/FacebookAI/roberta-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect" 

2026-04-14 04:26:14,834 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/FacebookAI/roberta-base/e2da8e2f811d1448a5b465c236feacd80ffbac7b/config.json "HTTP/1.1 200 OK" 

2026-04-14 04:26:14,936 | INFO | HTTP Request: HEAD https://huggingface.co/FacebookAI/roberta-base/resolve/main/model.safetensors "HTTP/1.1 302 Found" 

2026-04-14 04:26:15,037 | INFO | HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-base/xet-read-token/e2da8e2f811d1448a5b465c236feacd80ffbac7b "HTTP/1.1 200 OK" 



model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-04-14 04:26:18,890 | INFO | Epoch 1/4 | Train Loss: 8.0997 | Val Loss: 2.5959 

2026-04-14 04:26:44,807 | INFO | Checkpoint saved: /content/drive/MyDrive/UdS-IFT714-checkpoints/roberta_base_1_4_last.pt 

2026-04-

In [10]:
# torch.save(model.state_dict(), "outputs/checkpoints/distillbert_test.pt")
# temp to avoid retraining, needs to be put into training function

path = "outputs/results/preds.csv"
ensemble.to_csv(path)

ensemble_model = AverageEnsemble(path)
pred_v, pred_a, gold_v, gold_a = ensemble_model.predictions()

eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
model_results["average_ensemble"] = eval_score
trained_models["average_ensemble"] = ensemble_model


### Step 3 : Analyze results

In [11]:
with open("./outputs/results/metrics.yaml", "w") as f:
    # yaml.safe_dump(model_results, f)
    pass

In [12]:

logging.info("Récapitulatif des résultats:")
if "google.colab" in sys.modules :
    logging.info(f"Running in Colab with {device} device...")
for mod, scores in model_results.items():
    line = (
        f"- {mod} : "
        f"PCC_V = {scores['PCC_V']:.4f} | "
        f"PCC_A = {scores['PCC_A']:.4f} | "
        f"RMSE_V = {scores['RMSE_V']:.4f} | "
        f"RMSE_A = {scores['RMSE_A']:.4f}| "
        f"RMSE_VA = {scores['RMSE_VA']:.4f}"
    )
    logging.info(line)

2026-04-14 04:28:27,526 | INFO | Récapitulatif des résultats: 

2026-04-14 04:28:27,527 | INFO | Running in Colab with cuda device... 

2026-04-14 04:28:27,527 | INFO | - baby_bert : PCC_V = 0.6519 | PCC_A = 0.3662 | RMSE_V = 1.4510 | RMSE_A = 1.0731| RMSE_VA = 1.2761 

2026-04-14 04:28:27,528 | INFO | - bert_base : PCC_V = 0.4922 | PCC_A = 0.2908 | RMSE_V = 1.5707 | RMSE_A = 0.9243| RMSE_VA = 1.2887 

2026-04-14 04:28:27,528 | INFO | - deberta_base : PCC_V = 0.0518 | PCC_A = 0.0926 | RMSE_V = 2.0722 | RMSE_A = 1.1659| RMSE_VA = 1.6813 

2026-04-14 04:28:27,528 | INFO | - roberta_base : PCC_V = 0.7168 | PCC_A = -0.1186 | RMSE_V = 1.3474 | RMSE_A = 1.3286| RMSE_VA = 1.3381 

2026-04-14 04:28:27,529 | INFO | - average_ensemble : PCC_V = 0.6803 | PCC_A = 0.0563 | RMSE_V = 1.3443 | RMSE_A = 1.0757| RMSE_VA = 1.2174 



In [ ]:
# CTRL+S to commit main.ipynb and...
# but doesn't work anymore in organization repo...
if "google.colab" in sys.modules :
  from google.colab import userdata, _message
  from getpass import getpass

  try :
    resp = _message.blocking_request('get_ipynb', timeout_sec=5)
    if not resp or not isinstance(resp, dict):
        raise ValueError("Couldn't fetch Colab notebook to commit.")
    with open('main.ipynb', 'w') as f:
        json.dump(resp['ipynb'], f)
  except Exception as e:
     print(type(e).__name__, "-", e)

  # GitHub / Settings / Emails (look for 123+user@users.noreply.github.com)
  try:
    email = userdata.get("GITHUB_EMAIL")
  except Exception:
    email = input("Enter your email: ")
  !git config --global user.email {email}

  try:
    name = userdata.get("GITHUB_NAME")
  except Exception:
    name = input("Enter your email: ")
  !git config --global user.name {name}

  !git status
  print()

  !git add outputs/ main.ipynb
  !git commit -m "feat: auto colab outputs"
  print()

  # GitHub / Settings / Developer settings / Personal access tokens
  try:
    token = userdata.get("GITHUB_TOKEN")
  except Exception:
    token = getpass("Enter GitHub token: ")
  !git push "https://{token}@github.com/Projet-NLP-UdeS/NLP_semeval26_task3_DimASR.git"